### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code.

Tools are pairings of:

1. A **schema**, including the name of the tool, a description, and/or argument definitions (often a JSON schema).
2. A **function** or **coroutine** to execute.

In [3]:
import os
from langchain.chat_models import init_chat_model

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

model = init_chat_model("google_genai:gemini-2.5-flash")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots talk primarily due to a unique combination of their highly developed vocal learning abilities, social intelligence, and specific brain structures. It\'s a form of mimicry they use for social bonding, communication within their flock (or human family), and sometimes even for survival.\n\nHere\'s a breakdown of the key reasons:\n\n1.  **Vocal Learning:** Unlike most animals that have fixed vocalizations, parrots are "vocal learners." This means they can learn to imitate and produce new sounds throughout their lives, much like humans learn language. This ability is rare in the animal kingdom, shared by only a few groups like songbirds, hummingbirds, and bats.\n\n2.  **Social Bonding and Communication:**\n    *   **In the Wild:** In their natural habitats, parrots are highly social creatures that live in large flocks. Vocal learning is crucial for flock cohesion. They use distinct calls to identify individuals, warn of predators, find mates, and maintain contact 

In [4]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools([get_weather])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'cb737c66-f3a2-42b3-a9ff-041fafa2caa5': 'CvIBAQw51sdH8PZSaz1OQlVofeW9M9e8jFo83Yj6kQQQYfWipEzq713wbmtdktUfTZBxHhqrBXJzRen42fYUY4TtxOcF8RTJ9cP6m4vyiIjOEBGHOR0Xu13Dg+Gezfco6f7PXWPIWFee1HDuI91cM2keseiXrvn3ByTVcaulucQ4f5Dkak6dI172VcojE3tzBXds9k6bIDbXif4zdG1F0hzvKlrhhOFr5hb1CAtXQYPPhlrFp34qtP9VgZIDgXnCSWc138xNasIRMZGKVP2QreYo+OCFu+Y2yJzZwvfEpue0o/5k1ZhloLfOl5Wm9JlkLhOkPss='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f02c6-7b44-7d62-98d5-bb8db04c7b6c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'cb737c66-f3a2-42b3-a9ff-041fafa2caa5', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 48, 'output_tokens': 63, 'total_tokens': 111, 'input_token_details': {'cache_read': 0}, 'out

### Tool Execution Loop

In [6]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

It's sunny in Boston.


In [7]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Boston"}'}, '__gemini_function_call_thought_signatures__': {'98dbdc27-f301-4bd0-af9c-dafe0d09c314': 'CukBAQw51scNb9mnS90AJgIt+e0mz4/eR1mW4dleYdErHZazcxtCzrm/klM8fG6kjcUvo3FyqgakNJdeGgXWQsprRCA+/JrXtV/sxydb7m0wegSp+celCiLafUqEqOqdNpw/t+5qkqnKMMTzW/y0pWfOx7lz/mI7CNxFIShF+s1VEOjqY83HjngZrEGqVXfvV8fVvNiI8tIl3HMlJto3AwfHZfp/5KrGsMV6vpuL1c5+nMVExWzkphhE9rj8o1ucKkXuyOloRIV38I9HuaK+9XUo1j/JPi5zaPZbdMs6FaNJPl897F90PSHbQac='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f02db-7909-7141-96cc-6ac39fd4f448-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '98dbdc27-f301-4bd0-af9c-dafe0d09c314', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 61,